In [14]:
 #tool binding is the process of connecting a tool to an agent, 
 # allowing the agent to utilize the tool's functionality. 
 # This involves defining the tool's interface, specifying how the agent can invoke the tool, 
 # and ensuring that the agent can interpret the tool's outputs effectively.
 #  Tool binding enables agents to perform complex tasks by leveraging external tools, enhancing 
 # their capabilities and allowing for more dynamic interactions.

#basically tool binding connects tool and llm 
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
import os

os.environ['HF_HOME'] = 'D:/huggingface_cache'

llm = HuggingFacePipeline.from_model_id(
    model_id='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    task='text-generation',
    pipeline_kwargs=dict(
        temperature=0.5,
        max_new_tokens=100
    )
)






Device set to use mps:0


In [15]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [16]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b


print(multiply.invoke({"a":3, "b":5}))
print(multiply.name)
print(multiply.description)
print(multiply.args)
print(multiply.args_schema.model_json_schema())

15
multiply
Given 2 numbers a and b this tool returns their product
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
{'description': 'Given 2 numbers a and b this tool returns their product', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


In [17]:
#tool binding
chat = ChatHuggingFace(llm=llm)
llm_with_tools = chat.bind_tools([multiply])



In [18]:
query = HumanMessage('can you multiply 3 with 1000')
messages = [query]
result = llm_with_tools.invoke(messages)
messages.append(result)

In [ ]:
from langchain_core.messages import AIMessage

# in tool calling the llm only suggest the tool and its input does not execture the tool 
#tool exectuion is the process of actually running the tool's function with the provided inputs,
if result.tool_calls:
    tool_result = multiply.invoke(result.tool_calls[0]["args"])
else:
    tool_result = multiply.invoke({"a": 3, "b": 1000})
messages.append(tool_result)
# ensure tool result is a Message (not a raw int)
if not hasattr(tool_result, "content"):
    messages[-1] = AIMessage(content=str(tool_result))

# remove any raw tool outputs from the message history
messages = [m if hasattr(m, "content") else AIMessage(content=str(m)) for m in messages]

response = llm_with_tools.invoke(messages)
response.content


NotImplementedError: Unsupported message type: <class 'int'>
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/MESSAGE_COERCION_FAILURE 